# SNR Peak Identification on a Complex Gamma Spectrum

This notebook demonstrates peak domain identification on a ¹⁵²Eu gamma-ray spectrum — a source with many closely-spaced lines across a wide energy range (122–1408 keV). The SNR convolution method handles the varying detector resolution automatically, making it suitable for both isolated and near-overlapping peaks.

## Workflow
1. Load and calibrate the ¹⁵²Eu spectrum
2. Set up SNR convolution and peak finder
3. Find all peak domains and visualise them

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspectrum.core import Spectrum, Domain
from pyspectrum.calibration.detector_calibration import DetectorCalibration, StandardHPGeFWHMModel, PolynomialEnergyModel
from pyspectrum.identification import Convolution
from pyspectrum.identification.snr import SNRFinder
from pyspectrum.identification.kernels.mexican_hat import gaussian_2_dev

## 1. Load and calibrate the spectrum

> **Adapt:** replace `path`, `domains`, and `energies` with values for your calibration source. See the *Calibration* notebook for guidance on choosing peak windows.

In [ ]:
# ── Adapt to your source and detector ────────────────────────────────────────
path     = '../Library/152Eu_calsource_10cm_85ks.txt'
domains  = [(300,320),(1057,1082),(2544,2567),(3177,3202),(3597,3612),(3687,3707),(4687,4727)]
energies = [121.78, 344.28, 778.9, 964.08, 1085.837, 1112.076, 1408.0]   # keV
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(path, names=['counts'])
first_non_zero = np.where(df['counts'] > 0)[0][0]
df = df[first_non_zero:].reset_index(drop=True)
df['channel'] = df.index

spectrum = Spectrum.from_dataframe(df=df, channel_col='channel', counts_col='counts')

detector_calibration = DetectorCalibration(
    spectrum, known_axis_values=energies, peak_domains=domains,
    energy_model=PolynomialEnergyModel(2), fwhm_model=StandardHPGeFWHMModel(),
)
energy_calib, fwhm_calib = detector_calibration.generate(energy_p0=(0,1,0), fwhm_p0=(0,1,0))
spectrum.set_axis_calibration(energy_calib)
spectrum.set_resolution_calibration(fwhm_calib)

In [ ]:
spectrum.data.plot(yscale='log')
plt.title('¹⁵²Eu — calibrated spectrum')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.grid(True, which='both')
plt.tight_layout()
plt.show()

## 2. Set up SNR convolution and find peak domains

> **Adapt `n_sigma_signal_threshold`** to control sensitivity. For a dense ¹⁵²Eu spectrum, 4–5σ avoids spurious detections between closely-spaced lines.

In [ ]:
# ── Adapt detection thresholds ────────────────────────────────────────────────
n_sigma_signal     = 5.0
n_sigma_background = 2.5
persistence        = 1.0
# ─────────────────────────────────────────────────────────────────────────────

conv   = Convolution(resolution=spectrum.resolution_calib.apply,
                     kernel=gaussian_2_dev, window_fwhm=4)
finder = SNRFinder(convolution=conv,
                   n_sigma_signal_threshold=n_sigma_signal,
                   n_sigma_bg_threshold=n_sigma_background,
                   persistence_factor=persistence)
domains = finder.find(spectrum)
print(f"Detected {len(domains)} peak domains")

## 3. Visualise detected domains

In [ ]:
spectrum.data.plot(yscale='log', color='steelblue')
for domain in domains:
    domain.data.plot(color='tomato')   # highlight each detected domain in red

plt.title('¹⁵²Eu — detected peak domains (100–500 keV region)')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.xlim([100, 500])
plt.ylim([1e3, 2e6])
plt.grid(True, which='both')
plt.tight_layout()
plt.show()